# Student Performance Factors Project

This notebook covers Data Exploration, Data Quality Assessment, Data Cleaning,
and Feature Engineering for the `StudentPerformanceFactors_messy.csv` dataset.
The goal is to produce a clean, analysis-ready dataset (`student_cleaned.csv`)
that later notebooks (`student_visualization.ipynb`, `Student_model.ipynb`) build on.

## Import Libraries

In [1]:
# Import required libraries
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## Day 1: Data Exploration

In [2]:
# Load the dataset
data = pd.read_csv("StudentPerformanceFactors_messy.csv")

# Column names have stray whitespace (e.g. ' Exam_Score '); strip them so we can
# reference columns normally. The values inside columns are cleaned later, in Day 3.
data.columns = [c.strip() for c in data.columns]

# Display first 5 rows
data.head()

,Hours_Studied,attendance,Parental Involvement,Access_to_Resources,ExtracurricularActivities,Sleep_hours,Previous_Scores,motivation_level,Internet_Access,Tutoring_Sessions,Family Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23.0,66.0,High,High,Yes,7.0,80,Medium,Yes,1,Medium,Medium,public,Positive,3,No,College,Moderate,Female,68.0
1,16.0,75.0,Med,Medium,Yes,8.0,94,Medium,Yes,0,Medium,Medium,Private,Positive,3,No,High School,Moderate,Male,65.0
2,22.0,98.0,Medium,Medium,Yes,6.0,93,Medium,Yes,4,Low,Medium,Public,Neutral,4,No,High School,Near,Female,73.0
3,28.0,76.0,Medium,High,Yes,6.0,75,High,Yes,3,High,Medium,Public,Positive,3,No,High School,Near,Male,72.0
4,19.0,76.0,High,Low,Yes,9.0,78,Low,Yes,1,Low,Medium,Public,Negative,1,No,Postgraduate,Near,Female,65.0


In [3]:
# Check rows, columns, column names, and data types
print("Rows and Columns:", data.shape)
print("\nColumn Names:")
print(list(data.columns))

Rows and Columns: (6707, 20)

Column Names:
['Hours_Studied', 'attendance', 'Parental Involvement', 'Access_to_Resources', 'ExtracurricularActivities', 'Sleep_hours', 'Previous_Scores', 'motivation_level', 'Internet_Access', 'Tutoring_Sessions', 'Family Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence', 'Physical_Activity', 'Learning_Disabilities', 'Parental_Education_Level', 'Distance_from_Home', 'Gender', 'Exam_Score']


In [4]:
# Basic information
print("Dataset Information:")
data.info()

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 6707 entries, 0 to 6706
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Hours_Studied              6573 non-null   str    
 1   attendance                 6606 non-null   str    
 2   Parental Involvement       6506 non-null   str    
 3   Access_to_Resources        6707 non-null   str    
 4   ExtracurricularActivities  6707 non-null   str    
 5   Sleep_hours                6573 non-null   str    
 6   Previous_Scores            6707 non-null   int64  
 7   motivation_level           6539 non-null   str    
 8   Internet_Access            6707 non-null   str    
 9   Tutoring_Sessions          6707 non-null   int64  
 10  Family Income              6471 non-null   str    
 11  Teacher_Quality            6335 non-null   str    
 12  School_Type                6707 non-null   str    
 13  Peer_Influence             6707 non-nu

In [5]:
# Statistical summary of numerical columns
data.describe()

,Previous_Scores,Tutoring_Sessions,Physical_Activity,Exam_Score
count,6707.000000,6707.000000,6707.000000,6640.000000
mean,75.061428,1.492918,2.966900,67.259940
std,14.400313,1.229206,1.031899,3.992789
min,50.000000,0.000000,0.000000,55.000000
25%,63.000000,1.000000,2.000000,65.000000
50%,75.000000,1.000000,3.000000,67.000000
75%,88.000000,2.000000,4.000000,69.000000
max,100.000000,8.000000,6.000000,110.000000


In [6]:
# Unique values in important columns
important_cols = ["Parental Involvement", "School_Type", "Peer_Influence", "Gender"]
for col in important_cols:
    print(f"{col}: {data[col].astype(str).str.strip().unique()}")

Parental Involvement: <ArrowStringArray>
[  'High',    'Med', 'Medium',    'Low',   'HIGH',      nan, 'medium',
 'MEDIUM',    'LOW',   'high',    'low']
Length: 11, dtype: str
School_Type: <ArrowStringArray>
['public', 'Private', 'Public', 'PRIVATE', 'PUBLIC', 'private']
Length: 6, dtype: str
Peer_Influence: <ArrowStringArray>
['Positive',  'Neutral', 'Negative', 'NEGATIVE', 'POSITIVE', 'positive',
 'negative',  'NEUTRAL',  'neutral']
Length: 9, dtype: str
Gender: <ArrowStringArray>
['Female', 'Male', 'M', 'MALE', nan, 'male', 'FEMALE', 'F', 'female']
Length: 9, dtype: str


In [7]:
# Day 1 Answers
rows, columns = data.shape
unique_school_types = data["School_Type"].astype(str).str.strip().nunique()
avg_exam_score_raw = pd.to_numeric(data["Exam_Score"].astype(str).str.strip(), errors="coerce").mean()

print(f"Day 1 Answers")
print(f"1. The dataset has {rows} rows and {columns} columns.")
print(f"2. There are {unique_school_types} distinct raw values for School_Type (before cleaning).")
print(f"3. The average Exam Score (raw, uncleaned) is approximately {avg_exam_score_raw:.2f}.")

Day 1 Answers
1. The dataset has 6707 rows and 20 columns.
2. There are 6 distinct raw values for School_Type (before cleaning).
3. The average Exam Score (raw, uncleaned) is approximately 67.26.


## Day 2: Data Quality Assessment

In [8]:
# 1. Check missing values
missing_values = data.isnull().sum()
print("Missing Values per column:")
print(missing_values)

Missing Values per column:
Hours_Studied                134
attendance                   101
Parental Involvement         201
Access_to_Resources            0
ExtracurricularActivities      0
Sleep_hours                  134
Previous_Scores                0
motivation_level             168
Internet_Access                0
Tutoring_Sessions              0
Family Income                236
Teacher_Quality              372
School_Type                    0
Peer_Influence                 0
Physical_Activity              0
Learning_Disabilities          0
Parental_Education_Level     436
Distance_from_Home           332
Gender                        67
Exam_Score                    67
dtype: int64


In [9]:
# 2. Check duplicate records
duplicate_rows = data.duplicated().sum()

print("Duplicate Rows:", duplicate_rows)

Duplicate Rows: 3


In [10]:
# 3. Inspect numerical-looking columns (some are stored as text due to stray spaces)
num_like_cols = ["Hours_Studied", "attendance", "Sleep_hours", "Previous_Scores",
                  "Tutoring_Sessions", "Physical_Activity", "Exam_Score"]
for col in num_like_cols:
    coerced = pd.to_numeric(data[col].astype(str).str.strip(), errors="coerce")
    print(f"{col}: dtype={data[col].dtype}, min={coerced.min()}, max={coerced.max()}, "
          f"non-numeric entries={coerced.isna().sum() - data[col].isna().sum()}")

Hours_Studied: dtype=str, min=-2.0, max=526.0, non-numeric entries=2
attendance: dtype=str, min=-9.0, max=719.0, non-numeric entries=1
Sleep_hours: dtype=str, min=-3.0, max=931.0, non-numeric entries=1
Previous_Scores: dtype=int64, min=50, max=100, non-numeric entries=0
Tutoring_Sessions: dtype=int64, min=0, max=8, non-numeric entries=0
Physical_Activity: dtype=int64, min=0, max=6, non-numeric entries=0
Exam_Score: dtype=float64, min=55.0, max=110.0, non-numeric entries=0


In [11]:
# 4. Inspect important categorical columns
cat_cols = ["Parental Involvement", "Access_to_Resources", "Family Income",
            "Teacher_Quality", "School_Type", "Peer_Influence",
            "Parental_Education_Level", "Distance_from_Home", "Gender"]
for col in cat_cols:
    vals = [v for v in data[col].astype(str).str.strip().unique().tolist() if pd.notna(v) and str(v) != "nan"]
    print(f"{col} ({len(vals)} raw variants): {sorted(vals)[:10]} ...")

Parental Involvement (10 raw variants): ['HIGH', 'High', 'LOW', 'Low', 'MEDIUM', 'Med', 'Medium', 'high', 'low', 'medium'] ...
Access_to_Resources (10 raw variants): ['HIGH', 'High', 'LOW', 'Low', 'MEDIUM', 'Med', 'Medium', 'high', 'low', 'medium'] ...
Family Income (13 raw variants): ['-', '?', 'HIGH', 'High', 'LOW', 'Low', 'MEDIUM', 'Med', 'Medium', 'high'] ...
Teacher_Quality (13 raw variants): ['-', '?', 'HIGH', 'High', 'LOW', 'Low', 'MEDIUM', 'Med', 'Medium', 'high'] ...
School_Type (6 raw variants): ['PRIVATE', 'PUBLIC', 'Private', 'Public', 'private', 'public'] ...
Peer_Influence (9 raw variants): ['NEGATIVE', 'NEUTRAL', 'Negative', 'Neutral', 'POSITIVE', 'Positive', 'negative', 'neutral', 'positive'] ...
Parental_Education_Level (14 raw variants): ['-', '?', 'COLLEGE', 'College', 'HIGH SCHOOL', 'High School', 'Highschool', 'POSTGRADUATE', 'Post Graduate', 'Postgraduate'] ...
Distance_from_Home (12 raw variants): ['-', '?', 'FAR', 'Far', 'MODERATE', 'Moderate', 'NEAR', 'Near', '

In [12]:
# 5. Check for placeholder / inconsistent missing-value markers
placeholder_counts = {}
for col in cat_cols:
    s = data[col].astype(str).str.strip().str.lower()
    placeholder_counts[col] = s.isin(["-", "?", "none", "unknown", "nan", ""]).sum()
print("Placeholder missing-value counts by column:")
print(placeholder_counts)

Placeholder missing-value counts by column:
{'Parental Involvement': np.int64(0), 'Access_to_Resources': np.int64(0), 'Family Income': np.int64(30), 'Teacher_Quality': np.int64(26), 'School_Type': np.int64(0), 'Peer_Influence': np.int64(0), 'Parental_Education_Level': np.int64(29), 'Distance_from_Home': np.int64(34), 'Gender': np.int64(0)}


In [13]:
# Data Quality Report
print("DATA QUALITY REPORT")
print("=" * 50)

print("1. Missing values (native NaN):", int(missing_values.sum()))
print("2. Duplicate rows:", int(duplicate_rows))
print("3. Placeholder missing values ('-', '?', 'none', 'unknown'):",
      sum(placeholder_counts.values()))
print("4. Numeric columns stored as text due to stray whitespace:",
      "Hours_Studied, attendance, Sleep_hours, Exam_Score")
print("5. Inconsistent category spellings found in:", ", ".join(cat_cols))
print("6. Out-of-range values found (e.g. Sleep_hours of 504/-1, attendance of 719%)")

DATA QUALITY REPORT
1. Missing values (native NaN): 2248
2. Duplicate rows: 3
3. Placeholder missing values ('-', '?', 'none', 'unknown'): 119
4. Numeric columns stored as text due to stray whitespace: Hours_Studied, attendance, Sleep_hours, Exam_Score
5. Inconsistent category spellings found in: Parental Involvement, Access_to_Resources, Family Income, Teacher_Quality, School_Type, Peer_Influence, Parental_Education_Level, Distance_from_Home, Gender
6. Out-of-range values found (e.g. Sleep_hours of 504/-1, attendance of 719%)


## Day 3: Data Cleaning

In [14]:
# Create a copy for cleaning
student_clean = data.copy()

# Store values before cleaning, for a before/after comparison later
rows_before_cleaning = student_clean.shape[0]

# Clean column names (strip stray whitespace)
student_clean.columns = [c.strip() for c in student_clean.columns]

In [15]:
# 1. Strip whitespace from all text columns and unify missing-value placeholders
obj_cols = student_clean.select_dtypes(include=["object", "string"]).columns.tolist()

for col in obj_cols:
    student_clean[col] = student_clean[col].astype(str).str.strip()
    student_clean[col] = student_clean[col].replace({
        "nan": np.nan, "None": np.nan, "none": np.nan, "NONE": np.nan,
        "-": np.nan, "?": np.nan, "unknown": np.nan, "Unknown": np.nan,
        "UNKNOWN": np.nan, "": np.nan,
    })

print("Text columns cleaned:", obj_cols)

Text columns cleaned: ['Hours_Studied', 'attendance', 'Parental Involvement', 'Access_to_Resources', 'ExtracurricularActivities', 'Sleep_hours', 'motivation_level', 'Internet_Access', 'Family Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence', 'Learning_Disabilities', 'Parental_Education_Level', 'Distance_from_Home', 'Gender']


In [16]:
# 2. Fix numeric columns that were read in as text
numeric_cols = ["Hours_Studied", "attendance", "Sleep_hours", "Exam_Score"]
for col in numeric_cols:
    student_clean[col] = pd.to_numeric(student_clean[col], errors="coerce")

student_clean[numeric_cols].describe()

,Hours_Studied,attendance,Sleep_hours,Exam_Score
count,6571.000000,6605.000000,6572.00000,6640.000000
mean,20.067265,80.132475,7.24437,67.259940
std,8.639070,15.268866,13.02454,3.992789
min,-2.000000,-9.000000,-3.00000,55.000000
25%,16.000000,70.000000,6.00000,65.000000
50%,20.000000,80.000000,7.00000,67.000000
75%,24.000000,90.000000,8.00000,69.000000
max,526.000000,719.000000,931.00000,110.000000


In [17]:
# 3. Standardize categorical text (Med/medium/MEDIUM -> Medium, Y/yes -> Yes, etc.)
categorical_cols = [
    "Parental Involvement", "Access_to_Resources", "ExtracurricularActivities",
    "motivation_level", "Internet_Access", "Family Income", "Teacher_Quality",
    "School_Type", "Peer_Influence", "Learning_Disabilities",
    "Parental_Education_Level", "Distance_from_Home", "Gender",
]

value_map = {
    "high": "High", "med": "Medium", "medium": "Medium", "low": "Low",
    "yes": "Yes", "y": "Yes", "no": "No", "n": "No",
    "public": "Public", "private": "Private",
    "positive": "Positive", "neutral": "Neutral", "negative": "Negative",
    "college": "College", "high school": "High School", "highschool": "High School",
    "postgraduate": "Postgraduate", "post graduate": "Postgraduate",
    "near": "Near", "far": "Far", "moderate": "Moderate",
    "female": "Female", "male": "Male", "f": "Female", "m": "Male",
}

for col in categorical_cols:
    student_clean[col] = student_clean[col].str.lower().map(
        lambda x: value_map.get(x, x) if pd.notna(x) else x
    )

# Verify: each column should now only contain its canonical categories
for col in categorical_cols:
    print(col, "->", sorted(student_clean[col].dropna().unique().tolist()))

Parental Involvement -> ['High', 'Low', 'Medium']
Access_to_Resources -> ['High', 'Low', 'Medium']
ExtracurricularActivities -> ['No', 'Yes']
motivation_level -> ['High', 'Low', 'Medium']
Internet_Access -> ['No', 'Yes']
Family Income -> ['High', 'Low', 'Medium']
Teacher_Quality -> ['High', 'Low', 'Medium']
School_Type -> ['Private', 'Public']
Peer_Influence -> ['Negative', 'Neutral', 'Positive']
Learning_Disabilities -> ['No', 'Yes']
Parental_Education_Level -> ['College', 'High School', 'Postgraduate']
Distance_from_Home -> ['Far', 'Moderate', 'Near']
Gender -> ['Female', 'Male']


In [18]:
# 4. Detect and remove impossible / out-of-range values
def clip_to_nan(series, lo, hi):
    return series.where((series >= lo) & (series <= hi))

outlier_report = {}
for col, (lo, hi) in {
    "Sleep_hours": (0, 24), "attendance": (0, 100),
    "Hours_Studied": (0, 80), "Exam_Score": (0, 100),
}.items():
    before_na = student_clean[col].isna().sum()
    student_clean[col] = clip_to_nan(student_clean[col], lo, hi)
    after_na = student_clean[col].isna().sum()
    outlier_report[col] = after_na - before_na

print("Out-of-range values converted to NaN, per column:")
print(outlier_report)

Out-of-range values converted to NaN, per column:
{'Sleep_hours': np.int64(4), 'attendance': np.int64(5), 'Hours_Studied': np.int64(2), 'Exam_Score': np.int64(3)}


In [19]:
# 5. Handle missing values
# Rows missing the target (Exam_Score) can't be used, so drop them
student_clean = student_clean.dropna(subset=["Exam_Score"]).reset_index(drop=True)

numeric_features = ["Hours_Studied", "attendance", "Sleep_hours", "Previous_Scores",
                     "Tutoring_Sessions", "Physical_Activity"]
for col in numeric_features:
    student_clean[col] = student_clean[col].fillna(student_clean[col].median())

for col in categorical_cols:
    student_clean[col] = student_clean[col].fillna(student_clean[col].mode()[0])

print("Remaining missing values:", int(student_clean.isnull().sum().sum()))

Remaining missing values: 0


In [20]:
# 6. Remove duplicate records
duplicate_rows_found = student_clean.duplicated().sum()
student_clean = student_clean.drop_duplicates().reset_index(drop=True)

print("Duplicate rows removed:", int(duplicate_rows_found))

Duplicate rows removed: 73


In [21]:
# 7. Verify the cleaned dataset
print("Cleaned Dataset Information:")
student_clean.info()

Cleaned Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 6564 entries, 0 to 6563
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Hours_Studied              6564 non-null   float64
 1   attendance                 6564 non-null   float64
 2   Parental Involvement       6564 non-null   str    
 3   Access_to_Resources        6564 non-null   str    
 4   ExtracurricularActivities  6564 non-null   str    
 5   Sleep_hours                6564 non-null   float64
 6   Previous_Scores            6564 non-null   int64  
 7   motivation_level           6564 non-null   str    
 8   Internet_Access            6564 non-null   str    
 9   Tutoring_Sessions          6564 non-null   int64  
 10  Family Income              6564 non-null   str    
 11  Teacher_Quality            6564 non-null   str    
 12  School_Type                6564 non-null   str    
 13  Peer_Influence             656

In [22]:
# Data Cleaning Report
rows_after_cleaning = student_clean.shape[0]

print("DATA CLEANING REPORT")
print("=" * 50)
print(f"Rows before cleaning: {rows_before_cleaning}")
print(f"Rows after cleaning:  {rows_after_cleaning}")
print(f"Rows removed (missing target / duplicates): {rows_before_cleaning - rows_after_cleaning}")
print(f"Columns: {student_clean.shape[1]}")
print(f"Missing values remaining: {int(student_clean.isnull().sum().sum())}")

DATA CLEANING REPORT
Rows before cleaning: 6707
Rows after cleaning:  6564
Rows removed (missing target / duplicates): 143
Columns: 20
Missing values remaining: 0


## Day 4: Feature Engineering

In [23]:
# Create the Pass/Fail target used by the model notebook
PASS_THRESHOLD = 70
student_clean["Result"] = np.where(student_clean["Exam_Score"] >= PASS_THRESHOLD, "Pass", "Fail")

student_clean["Result"].value_counts()

Result
Fail    4948
Pass    1616
Name: count, dtype: int64

In [24]:
# Create a simple performance band feature (useful for grouped analysis/visuals)
def score_band(score):
    if score >= 90:
        return "Excellent"
    elif score >= 80:
        return "Good"
    elif score >= 70:
        return "Average"
    else:
        return "Needs Improvement"

student_clean["Performance_Band"] = student_clean["Exam_Score"].apply(score_band)

# Day 4 Questions and Answers
most_common_band = student_clean["Performance_Band"].mode()[0]
pass_rate = (student_clean["Result"] == "Pass").mean() * 100

print("Day 4 Answers")
print(f"1. The most common performance band is '{most_common_band}'.")
print(f"2. The overall pass rate (Exam_Score >= {PASS_THRESHOLD}) is {pass_rate:.1f}%.")

Day 4 Answers
1. The most common performance band is 'Needs Improvement'.
2. The overall pass rate (Exam_Score >= 70) is 24.6%.


In [25]:
# Day 4 Observations
print("DAY 4 OBSERVATIONS")
print("=" * 50)
print("1. The pass/fail split and performance bands give us convenient targets")
print("   for classification models in Student_model.ipynb.")
print("2. Hours_Studied, attendance, and Previous_Scores look like the strongest")
print("   numeric candidates for predicting Exam_Score (checked further in the")
print("   visualization notebook).")

DAY 4 OBSERVATIONS
1. The pass/fail split and performance bands give us convenient targets
   for classification models in Student_model.ipynb.
2. Hours_Studied, attendance, and Previous_Scores look like the strongest
   numeric candidates for predicting Exam_Score (checked further in the
   visualization notebook).


## Group by Analysis

In [26]:
# Average exam score by parental involvement
score_by_involvement = student_clean.groupby("Parental Involvement")["Exam_Score"].mean().sort_values(ascending=False)
print(score_by_involvement)

Parental Involvement
High      68.105177
Medium    67.104348
Low       66.356529
Name: Exam_Score, dtype: float64


In [27]:
# Average exam score by school type
score_by_school = student_clean.groupby("School_Type")["Exam_Score"].mean().sort_values(ascending=False)
print(score_by_school)

School_Type
Private    67.293440
Public     67.214364
Name: Exam_Score, dtype: float64


In [28]:
# Pass rate by teacher quality
pass_rate_by_teacher = student_clean.groupby("Teacher_Quality")["Result"].apply(
    lambda s: (s == "Pass").mean() * 100
).sort_values(ascending=False)
print(pass_rate_by_teacher)

Teacher_Quality
High      28.343423
Medium    23.647343
Low       20.098847
Name: Result, dtype: float64


In [29]:
# Average hours studied vs average exam score, by motivation level
motivation_summary = student_clean.groupby("motivation_level")[["Hours_Studied", "Exam_Score"]].mean()
print(motivation_summary)

                  Hours_Studied  Exam_Score
motivation_level                           
High                  19.743550   67.709148
Low                   19.959206   66.769189
Medium                20.092344   67.317943


In [30]:
# Final data quality check
print("Final Shape:", student_clean.shape)
print()
print(student_clean.isnull().sum().sum(), "missing values remaining")
student_clean.head()

Final Shape: (6564, 22)

0 missing values remaining


,Hours_Studied,attendance,Parental Involvement,Access_to_Resources,ExtracurricularActivities,Sleep_hours,Previous_Scores,motivation_level,Internet_Access,Tutoring_Sessions,Family Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score,Result,Performance_Band
0,23.0,66.0,High,High,Yes,7.0,80,Medium,Yes,1,Medium,Medium,Public,Positive,3,No,College,Moderate,Female,68.0,Fail,Needs Improvement
1,16.0,75.0,Medium,Medium,Yes,8.0,94,Medium,Yes,0,Medium,Medium,Private,Positive,3,No,High School,Moderate,Male,65.0,Fail,Needs Improvement
2,22.0,98.0,Medium,Medium,Yes,6.0,93,Medium,Yes,4,Low,Medium,Public,Neutral,4,No,High School,Near,Female,73.0,Pass,Average
3,28.0,76.0,Medium,High,Yes,6.0,75,High,Yes,3,High,Medium,Public,Positive,3,No,High School,Near,Male,72.0,Pass,Average
4,19.0,76.0,High,Low,Yes,9.0,78,Low,Yes,1,Low,Medium,Public,Negative,1,No,Postgraduate,Near,Female,65.0,Fail,Needs Improvement


In [31]:
# Export final cleaned dataset
student_clean.to_csv("student_cleaned.csv", index=False)
print("Saved cleaned dataset -> student_cleaned.csv")

Saved cleaned dataset -> student_cleaned.csv
